# 03 - Despliegue

Este notebook valida que el artefacto desplegable sea un pipeline completo y que acepte datos crudos pre-carrera. La fase de Evaluación ya ocurrió antes; aquí se documenta despliegue y monitoreo.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'app').exists() and (ROOT.parent / 'app').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / 'models' / 'best_model_pipe.pkl'
OUTPUT_DIR = ROOT / 'output'
pipe = joblib.load(MODEL_PATH)
print('Modelo cargado:', MODEL_PATH)
print('Pasos:', list(pipe.named_steps.keys()))

In [ ]:
with open(OUTPUT_DIR / 'feature_schema.json', encoding='utf-8') as f:
    schema = json.load(f)
print(json.dumps(schema, indent=2, ensure_ascii=False))

In [ ]:
raw_test = pd.read_csv(OUTPUT_DIR / 'test_unbalanced_raw.csv')
X_raw = raw_test.drop(columns=['finished'])
y_raw = raw_test['finished']

sample = X_raw.head(10)
pred = pipe.predict(sample)
prob = pipe.predict_proba(sample)[:, 1]
display(pd.DataFrame({'real': y_raw.head(10).to_numpy(), 'pred': pred, 'prob_finish': prob}).round(4))

In [ ]:
# Ejemplo manual con columnas crudas pre-carrera. No se envían variables escaladas ni derivadas.
example = X_raw.median(numeric_only=True).to_frame().T
example['grid'] = 1
example['driver_race_count'] = 120
example['constructor_race_count'] = 350
example['driver_prev_finish_rate'] = 0.85
example['constructor_prev_finish_rate'] = 0.82
example['driver_last5_finish_rate'] = 0.80
example['constructor_last5_finish_rate'] = 0.80
example['has_qualifying'] = 1
example['top10_start'] = 1

pred = pipe.predict(example)[0]
prob = pipe.predict_proba(example)[0, 1]
print(f'Predicción ejemplo crudo: pred={pred}, prob_finish={prob:.4f}')
display(example.T.rename(columns={0: 'valor'}))

## Streamlit

La aplicación `app/app.py` carga `models/best_model_pipe.pkl`. La interfaz debe enviar columnas crudas compatibles con `test_unbalanced_raw.csv`; el pipeline se encarga de eliminación de leakage, ingeniería, winsorización, selección, SMOTENC solo durante fit, escalado. En inferencia, el paso `sampler` no altera una muestra nueva porque `imblearn.Pipeline.predict` solo aplica transformaciones y el estimador final.